# Ridge Regression on Housing Data
Fitting a Ridge Regression model with hyperparameter tuning via GridSearchCV to predict median house values (MEDV).

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, RepeatedKFold, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

## 2. Load and Inspect Dataset

In [ ]:
df = pd.read_csv('housing.csv')
df.head()

In [ ]:
df.info()

## 3. Prepare Features and Target

In [ ]:
X = df.drop(['MEDV'], axis=1)
y = df['MEDV']

## 4. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

## 5. Baseline Ridge Model (alpha=1)

In [ ]:
ridge_model = Ridge(alpha=1).fit(X_train, y_train)

print('Intercept:', ridge_model.intercept_)

In [ ]:
y_pred = ridge_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f'RMSE : {rmse:.3f}')
print(f'R²   : {r2:.3f}')

In [ ]:
print('Coefficients:')
pd.Series(ridge_model.coef_, index=X_train.columns)

## 6. Hyperparameter Tuning with GridSearchCV

In [ ]:
cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=1)

# Define alpha grid
grid = {'alpha': np.arange(0, 1, 0.1)}

search = GridSearchCV(
    Ridge(),
    grid,
    scoring='neg_mean_absolute_error',
    cv=cv,
    n_jobs=-1
)

results = search.fit(X_train, y_train)

print('Best MAE  : %.3f' % results.best_score_)
print('Best Config:', results.best_params_)

## 7. Tuned Ridge Model (Best Alpha)

In [ ]:
best_alpha = results.best_params_['alpha']

ridge_tuned = Ridge(alpha=best_alpha).fit(X_train, y_train)
y_pred_tuned = ridge_tuned.predict(X_test)

rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
r2_tuned   = r2_score(y_test, y_pred_tuned)

print(f'Tuned RMSE : {rmse_tuned:.3f}')
print(f'Tuned R²   : {r2_tuned:.3f}')

In [ ]:
print('Tuned Model Coefficients:')
pd.Series(ridge_tuned.coef_, index=X_train.columns).sort_values()